In [5]:
!pip install sentence-transformers

In [6]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 39.2 MB/s eta 0:00:00


In [7]:
import re
import pandas as pd
import numpy as np

In [8]:
def clean_text(text):
  text = text.lower()
  text = re.sub(r'[^a-zA-Z0-9\s]','',text)
  text = re.sub(r'\s+',' ',text)
  return text

In [9]:
hotel_qna = [
    {"question": "What room types are available?", "answer": "We have Standard Rooms, Deluxe Rooms, Executive Suites, and Presidential Suites available."},
    {"question": "What are the room prices?", "answer": "Standard Room: $80/night, Deluxe Room: $150/night, Executive Suite: $250/night, Presidential Suite: $500/night."},
    {"question": "How can I book a room?", "answer": "You can book a room by calling us at +1-800-GRAND or visiting our website at www.grandpalacehotel.com."},
    {"question": "What is the check in time?", "answer": "Our check-in time is 2:00 PM. Early check-in is available upon request based on room availability."},
    {"question": "What is the check out time?", "answer": "Check-out time is 12:00 PM noon. Late check-out is available upon request."},
    {"question": "Do you have free wifi?", "answer": "Yes! We offer free high-speed WiFi throughout the hotel for all guests."},
    {"question": "Is there a swimming pool?", "answer": "Our outdoor swimming pool is open from 7:00 AM to 10:00 PM daily for all guests."},
    {"question": "Do you have a restaurant?", "answer": "Our in-house restaurant serves breakfast from 7 AM, lunch from 12 PM, and dinner from 7 PM."},
    {"question": "Is there a gym?", "answer": "Our fully equipped gym is open 24/7 for all guests. Personal trainers are available on request."},
    {"question": "Do you have a spa?", "answer": "Our luxury spa offers massages, facials, and beauty treatments. Open from 9 AM to 9 PM."},
    {"question": "Where is the hotel located?", "answer": "Grand Palace Hotel is located at 123 Luxury Avenue, Downtown City. 5 minutes from the airport."},
    {"question": "What is the contact number?", "answer": "You can reach us at +1-800-GRAND or email us at info@grandpalacehotel.com."},
    {"question": "What is the cancellation policy?", "answer": "Cancellations made 48 hours before check-in are free of charge. Late cancellations may incur a fee."},
    {"question": "Are pets allowed?", "answer": "We are a pet-friendly hotel! Pets are welcome with a small additional fee of $20 per night."},
    {"question": "Is parking available?", "answer": "We offer free parking for all hotel guests. Valet parking is also available."},
    {"question": "Do you provide airport transfer?", "answer": "Yes we provide free airport transfer for all our guests. Please inform us of your arrival time."},
    {"question": "What amenities does the hotel offer?", "answer": "We offer free WiFi, swimming pool, gym, spa, restaurant, room service, airport transfer and conference halls."},
    {"question": "Is breakfast included?", "answer": "Breakfast is included in Deluxe Room and above packages. Standard Room guests can add breakfast for $15/day."},
    {"question": "Do you have conference rooms?", "answer": "Yes we have fully equipped conference halls available for meetings and events. Please contact us for bookings."},
    {"question": "Is room service available?", "answer": "Yes! Room service is available 24/7 for all guests. Just call the front desk to place your order."}
]

df = pd.DataFrame(hotel_qna)
df['Cleaned_Question'] = df['question'].astype(str).apply(clean_text)
df.to_csv('hotel_qna.csv',index=False)
print(df[['question','Cleaned_Question']].head())

                         question               Cleaned_Question
0  What room types are available?  what room types are available
1       What are the room prices?       what are the room prices
2          How can I book a room?          how can i book a room
3      What is the check in time?      what is the check in time
4     What is the check out time?     what is the check out time


In [10]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
embeddings = model.encode(df['Cleaned_Question'].tolist())

In [12]:
embeddings = np.array(embeddings)

In [13]:
np.save('hotel_embeddings.npy',embeddings)
embeddings = np.load('hotel_embeddings.npy')

In [14]:
import faiss
dimensions = embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(dimensions)

In [15]:
faiss_index.add(embeddings)
faiss.write_index(faiss_index,'faiss_index.index')

In [16]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def get_similar_answer(query, count=1, model=model, faiss_index=faiss_index):
    query_embedding = model.encode([query])
    distances, indices = faiss_index.search(query_embedding, count)
    for i in range(count):
        print("Question:", df["question"].iloc[indices[0][i]])
        print("Answer:", df["answer"].iloc[indices[0][i]])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
get_similar_answer("Do you have a pool?")

Question: Is there a swimming pool?
Answer: Our outdoor swimming pool is open from 7:00 AM to 10:00 PM daily for all guests.


In [18]:
def chatbot_app():
    print(" -- Welcome to Grand Palace Hotel Bot -- ")
    print(" -- Type 'exit' to exit")
    while True:
        user_input = input("You: ")
        if user_input.lower() == "exit":
            print("ChatBot: Goodbye! Have a great stay!")
            break
        else:
            query_embedding = model.encode([user_input])
            distances, indices = faiss_index.search(query_embedding, 1)
            response = df["answer"].iloc[indices[0][0]]
            print("ChatBot:", response)

In [19]:
chatbot_app()

 -- Welcome to Grand Palace Hotel Bot -- 
 -- Type 'exit' to exit
You: whats your timing
ChatBot: Check-out time is 12:00 PM noon. Late check-out is available upon request.
You: exit
ChatBot: Goodbye! Have a great stay!
